# Analisis Kurva S Pengguna dan Merchant QRIS Indonesia 2021–2026

Notebook ini memuat seluruh kode analisis: pembangunan dataset, model logistik, Gompertz, Bass, dan bi-logistik, uji ketahanan, diagnostik residual, *moving block bootstrap*, gambar, serta pemeriksaan tambahan pola kurva S (laju pertumbuhan aktual, sensitivitas K, validasi *expanding-window*, dan grafik diagnostik).

**Cara pakai:** buka notebook ini dari folder utama repositori (sejajar dengan folder `data` dan `analisis`), lalu jalankan *Run All*. Hasil ditulis ke folder `data` dan `analisis`.

## 1. Persiapan

In [ ]:
# Pasang pustaka yang diperlukan (cukup sekali; boleh dilewati apabila sudah terpasang)
%pip install -q numpy pandas scipy matplotlib openpyxl

In [ ]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd

BASE = Path.cwd()                      # folder kerja: F:\QRIS
(BASE / "data").mkdir(exist_ok=True)
(BASE / "analisis").mkdir(exist_ok=True)
DATA_CSV = str(BASE / "data" / "qris_pengguna_merchant.csv")
ANALISIS = str(BASE / "analisis")
print("Folder kerja:", BASE)

## 2. Dataset QRIS (sumber: siaran pers BI dan ASPI)

Sel ini memuat seluruh titik data beserta status sumber, nomor siaran pers, tanggal, dan URL, lalu menulis `data/Dataset_QRIS_Indonesia_2021-2026.xlsx` dan empat berkas CSV. Untuk mengubah angka, ubah langsung di daftar `PM`, `TRX`, `GROWTH`, atau `X` di sel ini, lalu jalankan ulang notebook.

In [ ]:
"""Bangun Dataset_QRIS_Indonesia_2021-2026.xlsx dan CSV turunannya.

Satu sumber kebenaran: semua angka dan sumbernya ditulis di sini, lalu diekspor ke
Excel (dengan status verifikasi dan tautan) dan ke CSV yang dipakai skrip model.
Status sumber:
  RESMI BI    = angka tertulis langsung di siaran pers Bank Indonesia (bi.go.id)
  PEJABAT BI  = pernyataan pejabat BI (konferensi pers/acara), dikutip media
  ASPI        = data Asosiasi Sistem Pembayaran Indonesia, dikutip media
  TURUNAN     = dihitung dari angka resmi
"""
import csv, sys
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

OUT = str(BASE / "data")
BI = "https://www.bi.go.id/id/publikasi/ruang-media/news-release/Pages/{}.aspx"
BIEN = "https://www.bi.go.id/en/publikasi/ruang-media/news-release/Pages/{}.aspx"

# periode, pengguna (juta), merchant (juta), status, no. rilis, tanggal rilis, sumber, url, catatan
PM = [
    ("2021-12", 12.80, None, "TURUNAN", "", "", "BI via Bisnis.com 24/02/2023", "https://finansial.bisnis.com/read/20230224/90/1631430/cuan-bri-bbri-catatkan-nilai-transaksi-qris-rp74-triliun-sepanjang-2022", "28,75 juta (Des-2022) dikurangi tambahan 15,95 juta selama 2022"),
    ("2022-02", 15.99, None, "PEJABAT BI", "", "2022-04-04", "Kepala DKSP BI, paparan virtual, via Fortune IDN", "https://www.fortuneidn.com/finance/transaksi-digital-meningkat-bi-bidik-pengguna-qris-capai-26-juta-00-ccw2k-hclm40", ""),
    ("2022-09", 25.20, 21.60, "PEJABAT BI", "", "2022-10-28", "Asisten Gubernur BI via Kontan", "https://keuangan.kontan.co.id/news/makin-diminati-transaksi-qris-capai-rp-297-triliun-hingga-september-2022", ""),
    ("2022-12", 28.75, 22.70, "PEJABAT BI", "", "2023-02", "BI via Bisnis.com 24/02/2023", "https://finansial.bisnis.com/read/20230224/90/1631430/cuan-bri-bbri-catatkan-nilai-transaksi-qris-rp74-triliun-sepanjang-2022", "ASPI mencatat 23,97 juta merchant Des-2022 (Gusti, 2024)"),
    ("2023-02", 30.87, 24.90, "RESMI BI", "25/93/DKom", "2023-04-11", "Siaran Pers BI", BI.format("sp_259323"), ""),
    ("2023-05", 35.80, 26.10, "RESMI BI", "25/165/DKom", "2023-06-22", "Siaran Pers RDG Juni 2023", BIEN.format("sp_2516523"), "Posisi tersirat Mei 2023"),
    ("2023-06", None, 26.70, "RESMI BI", "25/211/DKom", "2023-08-03", "Siaran Pers BI (penghargaan QRIS)", BI.format("sp_2521123"), "91,4% merchant UMKM"),
    ("2023-07", 38.24, 27.51, "RESMI BI", "25/233/DKom", "2023-08-24", "Siaran Pers RDG Agustus 2023", BI.format("sp_2523323"), "Posisi tersirat Juli 2023"),
    ("2023-08", 40.05, 28.38, "RESMI BI", "25/259/DKom", "2023-09-21", "Siaran Pers RDG September 2023", BIEN.format("sp_2525923"), "Posisi tersirat Agustus 2023"),
    ("2023-09", 41.84, 29.04, "RESMI BI", "25/283/DKom", "2023-10-19", "Siaran Pers RDG Oktober 2023", BI.format("sp_2528323"), "Posisi triwulan III 2023"),
    ("2023-10", 43.44, 29.63, "RESMI BI", "25/317/DKom", "2023-11-23", "Siaran Pers RDG November 2023", BIEN.format("sp_2531723"), "Posisi tersirat Oktober 2023"),
    ("2023-11", 45.03, 30.12, "RESMI BI", "25/342/DKom", "2023-12-21", "Siaran Pers RDG Desember 2023", BI.format("sp_2534223"), ""),
    ("2023-12", 45.78, 30.41, "RESMI BI", "26/9/DKom", "2024-01-17", "Siaran Pers RDG Januari 2024", BI.format("sp_260924"), "Kinerja 2023"),
    ("2024-01", 46.37, 30.88, "RESMI BI", "26/33/DKom", "2024-02-21", "Siaran Pers RDG Februari 2024", BI.format("sp_263324"), ""),
    ("2024-02", 46.98, 31.27, "RESMI BI", "26/54/DKom", "2024-03-20", "Siaran Pers RDG Maret 2024", BI.format("sp_265424"), "Posisi tersirat Februari 2024"),
    ("2024-03", 48.12, 31.61, "RESMI BI", "26/80/DKom", "2024-04-24", "Siaran Pers RDG April 2024", BI.format("sp_268024"), "Posisi triwulan I 2024"),
    ("2024-04", 48.90, 31.86, "RESMI BI", "26/108/DKom", "2024-05-22", "Siaran Pers RDG Mei 2024", BI.format("sp_2610824"), ""),
    ("2024-05", 49.76, 32.25, "RESMI BI", "26/125/DKom", "2024-06-20", "Siaran Pers RDG Juni 2024", BIEN.format("sp_2612524"), ""),
    ("2024-06", 50.50, 32.71, "RESMI BI", "26/144/DKom", "2024-07-17", "Siaran Pers RDG Juli 2024", BIEN.format("sp_2614424"), "Posisi triwulan II 2024"),
    ("2024-08", 52.55, 33.77, "RESMI BI", "26/201/DKom", "2024-09-18", "Siaran Pers RDG September 2024", BI.format("sp_2620124"), ""),
    ("2024-09", 53.30, 34.23, "RESMI BI", "26/225/DKom", "2024-10-16", "Siaran Pers RDG Oktober 2024", BI.format("sp_2622524"), "Posisi triwulan III 2024 (versi sebelumnya memakai 53,0 dari media)"),
    ("2024-10", 54.10, 34.70, "RESMI BI", "26/254/DKom", "2024-11-20", "Siaran Pers RDG November 2024", BI.format("sp_2625424"), ""),
    ("2024-11", 55.02, 35.10, "RESMI BI", "26/275/DKom", "2024-12-18", "Siaran Pers RDG Desember 2024", BIEN.format("sp_2627524"), ""),
    ("2025-03", 56.30, 38.10, "PEJABAT BI", "", "2025-04-23", "Deputi Gubernur BI, konferensi pers RDG April 2025, via detik", "https://finance.detik.com/foto-bisnis/d-7889441/mantap-qris-tembus-56-juta-pengguna?zoom=1", "Rilis RDG 27/84/DKom hanya memuat pertumbuhan"),
    ("2025-06", 57.00, 39.30, "RESMI BI", "27/170/DKom", "2025-08-04", "Siaran Pers QRIS Jelajah Indonesia 2025", BI.format("sp_2717025"), "Posisi semester I 2025; 93,16% merchant UMKM"),
    ("2025-09", 58.00, 41.00, "PEJABAT BI", "", "2025-11-11", "Deputi Gubernur BI, Bulan Fintech Nasional, via Investortrust", "https://investortrust.id/financial/85155/tumbuh-158-transaksi-qris-tembus-10-33-miliar-per-september-2025", ""),
    ("2025-11", 59.00, 42.00, "PEJABAT BI", "", "2025-12-17", "Deputi Gubernur BI, konferensi pers RDG Desember 2025, via Bisnis.com", "https://ekonomi.bisnis.com/read/20251218/9/1937730/transaksi-qris-2-kali-lipat-dari-target-ekonomi-ri-cerah", "Disebut 'sampai dengan 2025'; posisi diasumsikan November 2025"),
    ("2026-01", 59.98, 43.06, "PEJABAT BI", "", "2026-03-12", "Deputi Gubernur BI via Kontan", "https://keuangan.kontan.co.id/news/bi-mencatat-nilai-transaksi-qris-capai-rp-16448-triliun-pada-januari-2026", "Pengguna +7,72% yoy, merchant +17,76% yoy"),
    ("2026-02", 60.77, None, "PEJABAT BI", "", "2026-04-08", "BI via Kontan", "https://keuangan.kontan.co.id/news/qris-cross-border-kian-diminati-transaksi-tumbuh-pesat", ""),
    ("2026-06", 65.77, 44.86, "RESMI BI", "28/159/DKom", "2026-08-17", "Siaran Pers Kartu Kredit Indonesia dan MDR 0%", BI.format("sp_2815926"), "6,23 juta pengguna baru Jan-Jun 2026; 96,68% merchant UMKM"),
    ("2026-07", None, 45.46, "RESMI BI", "28/191/DKom", "2026-09-22", "Siaran Pers QRIS Antarnegara 50-in-5 Awards", BI.format("sp_2819126"), "96,69% merchant UMKM"),
    ("2026-08", 69.32, 47.11, "RESMI BI", "28/196/DKom", "2026-09-24", "Siaran Pers FEKDI x IFSE 2026", BI.format("sp_2819626"), "Pengguna +20,26% yoy, merchant +13,83% yoy (yoy dari pernyataan Gubernur BI)"),
]

# periode, ukuran, volume (miliar trx), nilai (Rp triliun), status, no. rilis, sumber, url, catatan
TRX = [
    ("2020", "setahun", 0.12411, 8.21, "ASPI", "", "ASPI via RRI 18/04/2025", "https://www.rri.co.id/keuangan/1459019/pengguna-qris-makin-banyak-transaksi-capai-rp659-triliun", ""),
    ("2021", "setahun", 0.37469, 27.63, "ASPI", "", "ASPI via RRI 18/04/2025", "https://www.rri.co.id/keuangan/1459019/pengguna-qris-makin-banyak-transaksi-capai-rp659-triliun", ""),
    ("2022", "setahun", 1.00319, 99.98, "ASPI", "", "ASPI via RRI 18/04/2025", "https://www.rri.co.id/keuangan/1459019/pengguna-qris-makin-banyak-transaksi-capai-rp659-triliun", "BI resmi (25/211/DKom) menyebut 1,03 miliar transaksi"),
    ("2023", "setahun", 2.14, 226.00, "ASPI", "", "ASPI via RRI 18/04/2025", "https://www.rri.co.id/keuangan/1459019/pengguna-qris-makin-banyak-transaksi-capai-rp659-triliun", "BI resmi (26/9/DKom) menyebut nominal Rp229,96 triliun"),
    ("2024", "setahun", 6.24, 659.93, "ASPI", "", "ASPI via RRI 18/04/2025", "https://www.rri.co.id/keuangan/1459019/pengguna-qris-makin-banyak-transaksi-capai-rp659-triliun", "BI resmi (27/8/DKom) hanya menyebut pertumbuhan volume 175,2%"),
    ("2025", "setahun", 15.51, 1420.66, "PEJABAT BI", "", "Deputi Gubernur BI via Kontan (Mar 2026)", "https://keuangan.kontan.co.id/news/bi-mencatat-nilai-transaksi-qris-capai-rp-16448-triliun-pada-januari-2026", "Volume +148,54% yoy"),
    ("2025-H1", "Jan-Jun", 6.05, 579.00, "RESMI BI", "27/170/DKom", "Siaran Pers BI 04/08/2025", BI.format("sp_2717025"), ""),
    ("2026-H1", "Jan-Jun", 12.55, 1120.00, "RESMI BI", "28/159/DKom", "Siaran Pers BI 17/08/2026", BI.format("sp_2815926"), "Nilai Rp1,12 kuadriliun, +93,92% yoy"),
]

# periode, pertumbuhan yoy (%), basis, status, no. rilis, url
GROWTH = [
    ("2023-07", 84.50, "nominal", "25/233/DKom", BI.format("sp_2523323")),
    ("2023-08", 89.64, "nominal", "25/259/DKom", BIEN.format("sp_2525923")),
    ("2023-Q3", 87.90, "nominal", "25/283/DKom", BI.format("sp_2528323")),
    ("2023-10", 186.08, "nominal", "25/317/DKom", BIEN.format("sp_2531723")),
    ("2023-11", 157.43, "nominal", "25/342/DKom", BI.format("sp_2534223")),
    ("2023", 130.01, "nominal", "26/9/DKom", BI.format("sp_260924")),
    ("2024-01", 149.46, "nominal", "26/33/DKom", BI.format("sp_263324")),
    ("2024-02", 161.51, "nominal", "26/54/DKom", BI.format("sp_265424")),
    ("2024-Q1", 175.44, "nominal", "26/80/DKom", BI.format("sp_268024")),
    ("2024-04", 194.06, "nominal", "26/108/DKom", BI.format("sp_2610824")),
    ("2024-05", 213.31, "nominal", "26/125/DKom", BIEN.format("sp_2612524")),
    ("2024-Q2", 226.54, "tidak disebut", "26/144/DKom", BIEN.format("sp_2614424")),
    ("2024-08", 217.33, "tidak disebut", "26/201/DKom", BI.format("sp_2620124")),
    ("2024-Q3", 209.61, "tidak disebut", "26/225/DKom", BI.format("sp_2622524")),
    ("2024-10", 183.90, "tidak disebut", "26/254/DKom", BI.format("sp_2625424")),
    ("2024-11", 186.00, "volume", "26/275/DKom", BIEN.format("sp_2627524")),
    ("2024", 175.20, "volume", "27/8/DKom", BI.format("sp_270825")),
    ("2025-Q1", 169.15, "volume", "27/84/DKom", BI.format("sp_278425")),
    ("2025-05", 151.70, "volume", "27/133/DKom", BI.format("sp_2713325")),
    ("2025-07", 162.77, "volume", "27/194/DKom", BI.format("sp_2719425")),
    ("2025-08", 145.07, "volume", "27/219/DKom", BI.format("sp_2721925")),
    ("2025-10", 139.45, "tidak disebut", "27/274/DKom", BI.format("sp_2727425")),
    ("2026-05", 95.10, "volume", "28/126/DKom", BI.format("sp_2812626")),
    ("2026-Q2", 100.12, "tidak disebut", "28/142/DKom", BIEN.format("sp_2814226")),
    ("2026-08", 67.22, "volume", "28/193/DKom", BI.format("sp_2819326")),
]

# koridor, mulai, transaksi, nilai (Rp miliar), posisi, status, no. rilis, sumber, url
X = [
    ("Thailand", "2022-08", 994890, 437.54, "2025-06", "RESMI BI", "27/193/DKom", "Siaran Pers QRIS di Jepang", BI.format("sp_2719325")),
    ("Malaysia", "2023-05", 4310000, 1150.00, "2025-06", "RESMI BI", "27/193/DKom", "Siaran Pers QRIS di Jepang", BI.format("sp_2719325")),
    ("Singapura", "2023-11", 238216, 77.06, "2025-06", "RESMI BI", "27/193/DKom", "Siaran Pers QRIS di Jepang", BI.format("sp_2719325")),
    ("Thailand", "2022-08", 1640000, 656.27, "2026-02", "PEJABAT BI", "", "BI via Kontan 08/04/2026", "https://keuangan.kontan.co.id/news/qris-cross-border-kian-diminati-transaksi-tumbuh-pesat"),
    ("Singapura", "2023-11", 554510, 179.28, "2026-02", "PEJABAT BI", "", "BI via Kontan 08/04/2026", "https://keuangan.kontan.co.id/news/qris-cross-border-kian-diminati-transaksi-tumbuh-pesat"),
    ("Jepang (outbound)", "2025-08", 5088, 0.4288, "2026-02", "PEJABAT BI", "", "BI via Kontan 08/04/2026", "https://keuangan.kontan.co.id/news/qris-cross-border-kian-diminati-transaksi-tumbuh-pesat"),
    ("Total inbound", "2022-08", 21000000, 5900.00, "2026-07", "RESMI BI", "28/191/DKom", "Siaran Pers QRIS Antarnegara", BI.format("sp_2819126")),
    ("Total outbound", "2022-08", 4670000, 1730.00, "2026-07", "RESMI BI", "28/191/DKom", "Siaran Pers QRIS Antarnegara", BI.format("sp_2819126")),
]

HEAD = Font(name="Arial", bold=True, color="FFFFFF")
FILL = PatternFill("solid", fgColor="1F4E79")
BODY = Font(name="Arial", size=10)
LINK = Font(name="Arial", size=10, color="0563C1", underline="single")
STATUS_FILL = {"RESMI BI": "E2EFDA", "PEJABAT BI": "FFF2CC", "ASPI": "DDEBF7", "TURUNAN": "EDEDED"}

def sheet(ws, headers, rows, url_col=None, status_col=None, widths=None):
    ws.append(headers)
    for c in ws[1]:
        c.font, c.fill, c.alignment = HEAD, FILL, Alignment(wrap_text=True, vertical="center")
    for r in rows:
        ws.append(list(r))
    for row in ws.iter_rows(min_row=2):
        for c in row:
            c.font = BODY
        if url_col is not None and row[url_col].value:
            row[url_col].hyperlink = row[url_col].value; row[url_col].font = LINK
        if status_col is not None:
            s = row[status_col].value
            if s in STATUS_FILL:
                row[status_col].fill = PatternFill("solid", fgColor=STATUS_FILL[s])
    for i, w in enumerate(widths or [], 1):
        ws.column_dimensions[get_column_letter(i)].width = w
    ws.freeze_panes = "B2"

wb = Workbook()
ws = wb.active; ws.title = "Pengguna_Merchant"
sheet(ws, ["Periode (posisi)", "Pengguna (juta)", "Merchant (juta)", "Pengguna per merchant", "Status sumber",
           "No. siaran pers", "Tanggal rilis", "Sumber", "URL", "Catatan"],
      [(p, u, m, None, s, n, d, src, url, note) for p, u, m, s, n, d, src, url, note in PM],
      url_col=8, status_col=4, widths=[14, 14, 14, 14, 13, 14, 12, 46, 50, 50])
for i in range(2, len(PM) + 2):
    ws[f"D{i}"] = f'=IF(AND(ISNUMBER(B{i}),ISNUMBER(C{i})),B{i}/C{i},"")'
    ws[f"D{i}"].number_format = "0.00"
    for col in "BC":
        ws[f"{col}{i}"].number_format = "0.00"

ws2 = wb.create_sheet("Transaksi")
sheet(ws2, ["Periode", "Cakupan", "Volume (miliar transaksi)", "Nilai (Rp triliun)", "Nilai rata-rata per transaksi (Rp ribu)",
            "Status sumber", "No. siaran pers", "Sumber", "URL", "Catatan"],
      [(p, c, v, n, None, s, no, src, url, note) for p, c, v, n, s, no, src, url, note in TRX],
      url_col=8, status_col=5, widths=[11, 10, 14, 14, 16, 13, 14, 38, 50, 50])
for i in range(2, len(TRX) + 2):
    ws2[f"E{i}"] = f"=IF(C{i}>0,D{i}/C{i},\"\")"
    ws2[f"E{i}"].number_format = "0.0"

ws3 = wb.create_sheet("Pertumbuhan_RDG")
sheet(ws3, ["Periode", "Pertumbuhan transaksi QRIS (% yoy)", "Basis", "Status sumber", "No. siaran pers", "URL"],
      [(p, g, b, "RESMI BI", n, url) for p, g, b, n, url in GROWTH],
      url_col=5, status_col=3, widths=[11, 16, 14, 13, 14, 60])

ws4 = wb.create_sheet("Antarnegara")
sheet(ws4, ["Koridor", "Mulai", "Transaksi kumulatif", "Nilai kumulatif (Rp miliar)", "Posisi data", "Status sumber",
            "No. siaran pers", "Sumber", "URL"], X, url_col=8, status_col=5,
      widths=[18, 10, 16, 16, 11, 13, 14, 32, 60])
ws4.append([]); r = ws4.max_row + 1
ws4[f"A{r}"] = "Total antarnegara (inbound + outbound, s.d. Jul 2026)"
ws4[f"C{r}"] = "=C8+C9"; ws4[f"D{r}"] = "=D8+D9"
ws4[f"A{r+1}"] = "Porsi inbound (transaksi)"; ws4[f"C{r+1}"] = "=C8/(C8+C9)"; ws4[f"C{r+1}"].number_format = "0.0%"
ws4[f"A{r+2}"] = "Surplus nilai inbound - outbound (Rp miliar)"; ws4[f"D{r+2}"] = "=D8-D9"
for rr in range(r, r + 3):
    for c in ws4[rr]:
        c.font = Font(name="Arial", size=10, bold=True)

ws5 = wb.create_sheet("Keterangan")
info = [
    ["Dataset QRIS Indonesia 2021–2026"],
    ["Penyusun: Ade Rumintarsih, Universitas Nusa Mandiri. Disusun 12 Oktober 2026."],
    [""],
    ["Status sumber", "Arti"],
    ["RESMI BI", "Angka tertulis langsung di siaran pers Bank Indonesia (bi.go.id). Paling kuat untuk dikutip."],
    ["PEJABAT BI", "Pernyataan Gubernur/Deputi Gubernur/pejabat BI di konferensi pers atau acara, dikutip media. Kutip media dan sebut pejabatnya."],
    ["ASPI", "Data Asosiasi Sistem Pembayaran Indonesia yang dikutip media."],
    ["TURUNAN", "Dihitung dari angka resmi (cara hitung ada di kolom Catatan)."],
    [""],
    ["Catatan penting"],
    ["1. Tabel Statistik Sistem Pembayaran dan Infrastruktur Pasar Keuangan (SPIP) BI tidak memuat baris QRIS; sumber resmi utama adalah siaran pers BI."],
    ["2. Sejak 2025, siaran pers RDG hanya memuat pertumbuhan transaksi QRIS, bukan jumlah pengguna/merchant; angka 2025–2026 diambil dari siaran pers tematik BI atau pernyataan pejabat BI."],
    ["3. Siaran pers RDG tidak selalu menyebut bulan posisi; posisi ditetapkan dari konteks paragraf (lihat kolom Catatan)."],
    ["4. Angka pengguna adalah pengguna terdaftar, bukan pengguna aktif."],
    ["5. Kolom D (Pengguna_Merchant) dan E (Transaksi) serta ringkasan Antarnegara adalah rumus; ubah angka di kolom input dan hasilnya ikut berubah."],
]
for row in info:
    ws5.append(row)
ws5["A1"].font = Font(name="Arial", bold=True, size=13)
for row in ws5.iter_rows(min_row=2):
    for c in row:
        if c.font.size != 13:
            c.font = BODY
for row in (4, 10):
    ws5[f"A{row}"].font = Font(name="Arial", bold=True)
for s, f in STATUS_FILL.items():
    for rr in range(5, 9):
        if ws5[f"A{rr}"].value == s:
            ws5[f"A{rr}"].fill = PatternFill("solid", fgColor=f)
ws5.column_dimensions["A"].width = 22; ws5.column_dimensions["B"].width = 110

wb.move_sheet("Keterangan", offset=-4)
wb.active = 0
wb.save(f"{OUT}/Dataset_QRIS_Indonesia_2021-2026.xlsx")

with open(f"{OUT}/qris_pengguna_merchant.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f); w.writerow(["periode", "pengguna_juta", "merchant_juta", "status", "sumber"])
    for p, u, m, s, n, d, src, url, note in PM:
        w.writerow([p, "" if u is None else u, "" if m is None else m, s, f"{src} {n} {url}".strip()])
with open(f"{OUT}/qris_transaksi_tahunan.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f); w.writerow(["periode", "cakupan", "volume_miliar_trx", "nilai_triliun_rp", "status", "sumber"])
    for p, c, v, n, s, no, src, url, note in TRX:
        w.writerow([p, c, v, n, s, f"{src} {no} {url}".strip()])
with open(f"{OUT}/qris_antarnegara.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f); w.writerow(["koridor", "mulai", "transaksi", "nilai_miliar_rp", "posisi", "status", "sumber"])
    for k, m, t, v, pos, s, no, src, url in X:
        w.writerow([k, m, t, v, pos, s, f"{src} {no} {url}".strip()])
with open(f"{OUT}/qris_pertumbuhan_rdg.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f); w.writerow(["periode", "pertumbuhan_yoy_persen", "basis", "no_siaran_pers", "url"])
    for row in GROWTH:
        w.writerow(row)
print("pengguna:", sum(1 for r in PM if r[1] is not None), "merchant:", sum(1 for r in PM if r[2] is not None),
      "resmi pengguna:", sum(1 for r in PM if r[1] is not None and r[3] == "RESMI BI"),
      "resmi merchant:", sum(1 for r in PM if r[2] is not None and r[3] == "RESMI BI"))

In [ ]:
pm = pd.read_csv(DATA_CSV)
print("Jumlah titik pengguna:", pm.pengguna_juta.notna().sum(), "| merchant:", pm.merchant_juta.notna().sum())
print(pm.status.value_counts().to_string())
pm.tail(6)

## 3. Model logistik, Gompertz, dan Bass

Estimasi kuadrat terkecil nonlinier, *residual bootstrap* 1.000 ulangan untuk selang kepercayaan batas jenuh *K*, uji *holdout* (latih s.d. Juni 2025), penetrasi, proyeksi 2026–2030, dan dekomposisi margin ekstensif/intensif.

In [ ]:
"""Model difusi kurva S untuk pengguna dan merchant QRIS.

Model: logistik, Gompertz, Bass. Waktu t = bulan sejak Januari 2020 (QRIS wajib berlaku).
Keluaran: parameter, batas jenuh K (selang kepercayaan bootstrap), titik belok,
uji holdout (latih s.d. Juni 2025, uji Juli 2025 ke atas), dan proyeksi 2026-2030.
"""
import sys, json
import numpy as np, pandas as pd
from scipy.optimize import curve_fit

DATA = DATA_CSV
OUT = ANALISIS
rng = np.random.default_rng(2026)

df = pd.read_csv(DATA)
y_, m_ = df.periode.str.split("-", expand=True).astype(int).T.values
df["t"] = (y_ - 2020) * 12 + (m_ - 1)

def logistic(t, K, r, tm):
    return K / (1 + np.exp(-r * (t - tm)))

def gompertz(t, K, r, tm):
    return K * np.exp(-np.exp(-r * (t - tm)))

def bass(t, m, p, q):
    e = np.exp(-(p + q) * t)
    return m * (1 - e) / (1 + (q / p) * e)

MODELS = {
    "Logistik": (logistic, lambda K0: [K0, 0.06, 30], ([1, 1e-4, -60], [1000, 1, 200])),
    "Gompertz": (gompertz, lambda K0: [K0, 0.04, 20], ([1, 1e-4, -60], [1000, 1, 200])),
    "Bass": (bass, lambda K0: [K0, 0.005, 0.05], ([1, 1e-6, 1e-4], [1000, 0.5, 1])),
}

def fit(f, p0, bounds, t, y):
    p, _ = curve_fit(f, t, y, p0=p0, bounds=bounds, maxfev=200000)
    return p

def metrics(y, yhat, k):
    n = len(y); sse = float(np.sum((y - yhat) ** 2))
    return {"RMSE": float(np.sqrt(sse / n)), "MAPE_%": float(np.mean(np.abs((y - yhat) / y)) * 100),
            "AIC": float(n * np.log(sse / n) + 2 * k), "R2": float(1 - sse / np.sum((y - y.mean()) ** 2))}

def to_label(t):
    t = float(t); yr = 2020 + int(t // 12); mo = int(round(t % 12)) + 1
    if mo == 13: yr, mo = yr + 1, 1
    return f"{yr}-{mo:02d}"

hasil = {}
proj_rows = []
for kol, nama in [("pengguna_juta", "Pengguna"), ("merchant_juta", "Merchant")]:
    d = df[["t", "periode", kol]].dropna()
    t, y = d.t.values.astype(float), d[kol].values.astype(float)
    hasil[nama] = {"n": int(len(y)), "periode": f"{d.periode.iloc[0]} s.d. {d.periode.iloc[-1]}", "model": {}}
    for mn, (f, p0f, bnd) in MODELS.items():
        p = fit(f, p0f(y.max() * 1.3), bnd, t, y)
        yhat = f(t, *p)
        r = {"param": [float(v) for v in p], **metrics(y, yhat, 3)}
        # titik belok
        if mn == "Logistik": tb, nb = p[2], p[0] / 2
        elif mn == "Gompertz": tb, nb = p[2], p[0] / np.e
        else:
            mm, pp, qq = p
            tb = np.log(qq / pp) / (pp + qq) if qq > pp else 0.0
            nb = bass(tb, *p)
        r["titik_belok"] = to_label(tb); r["nilai_di_titik_belok"] = float(nb)
        r["K"] = float(p[0])
        # bootstrap residual
        res = y - yhat; Ks = []
        for _ in range(1000):
            yb = yhat + rng.choice(res, size=len(res), replace=True)
            try:
                Ks.append(fit(f, p, bnd, t, yb)[0])
            except Exception:
                pass
        r["K_CI95"] = [float(np.percentile(Ks, 2.5)), float(np.percentile(Ks, 97.5))]
        r["penetrasi_terhadap_K_%"] = float(y[-1] / p[0] * 100)
        # holdout: latih s.d. 2025-06 (t=65)
        tr = t <= 65
        if tr.sum() >= 5 and (~tr).sum() >= 1:
            ph = fit(f, p, bnd, t[tr], y[tr])
            yh = f(t[~tr], *ph)
            r["holdout_MAPE_%"] = float(np.mean(np.abs((y[~tr] - yh) / y[~tr])) * 100)
            r["holdout_bias_%"] = float(np.mean((yh - y[~tr]) / y[~tr]) * 100)
            r["holdout_K"] = float(ph[0])
        # proyeksi Desember tiap tahun
        for yr in range(2026, 2031):
            tt = (yr - 2020) * 12 + 11
            proj_rows.append({"variabel": nama, "model": mn, "tahun": yr, "proyeksi_juta": round(float(f(tt, *p)), 2)})
        hasil[nama]["model"][mn] = r

# Dekomposisi pertumbuhan volume transaksi: margin ekstensif (pengguna) vs intensif (trx/pengguna)
vol = {2022: 1.00319, 2023: 2.14, 2024: 6.24, 2025: 15.51}
usr = {2022: 28.75, 2023: 45.78, 2024: 55.02, 2025: 59.00}
dek = []
for a, b in [(2022, 2023), (2023, 2024), (2024, 2025), (2022, 2025)]:
    gv = np.log(vol[b] / vol[a]); gu = np.log(usr[b] / usr[a])
    dek.append({"periode": f"{a}-{b}", "pertumbuhan_volume_log": round(gv, 3),
                "porsi_pengguna_%": round(gu / gv * 100, 1), "porsi_intensitas_%": round((gv - gu) / gv * 100, 1)})
hasil["dekomposisi_volume"] = dek

json.dump(hasil, open(f"{OUT}/hasil_model.json", "w"), indent=2, ensure_ascii=False)
pd.DataFrame(proj_rows).to_csv(f"{OUT}/proyeksi_2026_2030.csv", index=False)
print(json.dumps(hasil, indent=1, ensure_ascii=False))

## 4. Model bi-logistik (dua gelombang)

In [ ]:
"""Uji gelombang kedua adopsi pengguna QRIS dengan model bi-logistik (Meyer, 1994).

N(t) = K1/(1+exp(-r1(t-tm1))) + K2/(1+exp(-r2(t-tm2)))
Dibandingkan dengan logistik tunggal melalui AIC dan residual titik terakhir.
"""
import sys, json
import numpy as np, pandas as pd
from scipy.optimize import curve_fit

df = pd.read_csv(DATA_CSV); OUT = ANALISIS
yy, mm = df.periode.str.split("-", expand=True).astype(int).T.values
df["t"] = (yy - 2020) * 12 + (mm - 1)
d = df[["t", "periode", "pengguna_juta"]].dropna()
t, y = d.t.values.astype(float), d.pengguna_juta.values

def lg(t, K, r, tm): return K / (1 + np.exp(-r * (t - tm)))
def bilg(t, K1, r1, tm1, K2, r2, tm2): return lg(t, K1, r1, tm1) + lg(t, K2, r2, tm2)

def aic(y, yh, k):
    n = len(y); sse = np.sum((y - yh) ** 2); return float(n * np.log(sse / n) + 2 * k)
def aicc(y, yh, k):
    n = len(y); return aic(y, yh, k) + 2 * k * (k + 1) / (n - k - 1)

p1, _ = curve_fit(lg, t, y, p0=[63, .09, 37], maxfev=100000)
lo = [40, .01, 10, 2, .05, 70]; hi = [80, 1, 60, 60, 1.5, 90]
p2, cov = curve_fit(bilg, t, y, p0=[59, .09, 36, 8, .3, 74], bounds=(lo, hi), maxfev=200000)
y1, y2 = lg(t, *p1), bilg(t, *p2)

rng = np.random.default_rng(7); res = y - y2; tot = []
for _ in range(1000):
    try:
        pb, _ = curve_fit(bilg, t, y2 + rng.choice(res, len(res)), p0=p2, bounds=(lo, hi), maxfev=50000)
        tot.append(pb[0] + pb[3])
    except Exception:
        pass

lab = lambda tt: f"{2020 + int(tt // 12)}-{int(tt % 12) + 1:02d}"
out = {
    "logistik_tunggal": {"param_K_r_tm": [float(v) for v in p1], "K": float(p1[0]), "AIC": aic(y, y1, 3), "AICc": aicc(y, y1, 3),
                         "residual_2026-06": float(y[-1] - y1[-1])},
    "bi_logistik": {"param_K1_r1_tm1_K2_r2_tm2": [float(v) for v in p2], "K1_gelombang1": float(p2[0]), "titik_belok_1": lab(p2[2]),
                    "K2_gelombang2": float(p2[3]), "titik_belok_2": lab(p2[5]),
                    "K_total": float(p2[0] + p2[3]),
                    "K_total_CI95": [float(np.percentile(tot, 2.5)), float(np.percentile(tot, 97.5))],
                    "AIC": aic(y, y2, 6), "AICc": aicc(y, y2, 6),
                    "residual_2026-06": float(y[-1] - y2[-1]),
                    "proyeksi_Des2026": float(bilg(83, *p2)), "proyeksi_Des2027": float(bilg(95, *p2)),
                    "proyeksi_Des2030": float(bilg(131, *p2))},
    "catatan": "Gelombang kedua didukung titik data 2026 (Jan, Feb, Jun, Agu); K2 dan K_total bersifat indikatif.",
}
json.dump(out, open(f"{OUT}/hasil_bilogistik.json", "w"), indent=2)
print(json.dumps(out, indent=1))

## 5. Uji ketahanan: hanya titik resmi siaran pers BI

In [ ]:
"""Uji ketahanan: estimasi ulang hanya dengan titik berstatus RESMI BI (siaran pers bi.go.id)."""
import sys, json
import numpy as np, pandas as pd
from scipy.optimize import curve_fit

df = pd.read_csv(DATA_CSV); OUT = ANALISIS
df = df[df.status == "RESMI BI"].copy()
yy, mm = df.periode.str.split("-", expand=True).astype(int).T.values
df["t"] = (yy - 2020) * 12 + (mm - 1)
lg = lambda t, K, r, tm: K / (1 + np.exp(-r * (t - tm)))
gp = lambda t, K, r, tm: K * np.exp(-np.exp(-r * (t - tm)))
bl = lambda t, K1, r1, tm1, K2, r2, tm2: lg(t, K1, r1, tm1) + lg(t, K2, r2, tm2)
def aicc(y, yh, k):
    n = len(y); sse = np.sum((y - yh) ** 2); return float(n * np.log(sse / n) + 2 * k + 2 * k * (k + 1) / (n - k - 1))
out = {}
for col, nama in [("pengguna_juta", "Pengguna"), ("merchant_juta", "Merchant")]:
    d = df.dropna(subset=[col]); t = d.t.values.astype(float); y = d[col].values
    r = {"n": int(len(y)), "periode": f"{d.periode.iloc[0]} s.d. {d.periode.iloc[-1]}"}
    p, _ = curve_fit(lg, t, y, p0=[y.max() * 1.3, .06, 40], maxfev=100000)
    r["Logistik"] = {"K": float(p[0]), "AICc": aicc(y, lg(t, *p), 3)}
    p, _ = curve_fit(gp, t, y, p0=[y.max() * 1.3, .04, 30], maxfev=100000)
    r["Gompertz"] = {"K": float(p[0]), "AICc": aicc(y, gp(t, *p), 3)}
    if nama == "Pengguna":
        p, _ = curve_fit(bl, t, y, p0=[60, .1, 36, 15, .45, 77], bounds=([40, .01, 10, 2, .05, 70], [80, 1, 60, 60, 1.5, 90]), maxfev=200000)
        r["Bi-logistik"] = {"K1": float(p[0]), "K2": float(p[3]), "K_total": float(p[0] + p[3]), "AICc": aicc(y, bl(t, *p), 6),
                            "proyeksi_Des2026": float(bl(83, *p))}
    out[nama] = r
json.dump(out, open(f"{OUT}/hasil_uji_ketahanan_resmi.json", "w"), indent=2)
print(json.dumps(out, indent=1))

## 6. Diagnostik residual

Shapiro-Wilk (normalitas), Durbin-Watson dan Ljung-Box *Q*(3) (autokorelasi), Breusch-Pagan (heteroskedastisitas), *runs test* Wald-Wolfowitz, dan uji F model bersarang (logistik tunggal vs bi-logistik).

In [ ]:
"""Diagnostik residual model difusi QRIS.

Uji: Shapiro-Wilk (normalitas), Durbin-Watson dan Ljung-Box (autokorelasi),
Breusch-Pagan (heteroskedastisitas terhadap nilai prediksi), runs test Wald-Wolfowitz
(pola sistematis tanda residual), dan uji F jumlah kuadrat sisa tambahan
(logistik tunggal vs bi-logistik, model bersarang).
Pemakaian: python diagnostik_residual.py data/qris_pengguna_merchant.csv analisis
"""
import sys, json
import numpy as np, pandas as pd
from scipy import stats
from scipy.optimize import curve_fit


# Implementasi langsung (numpy/scipy) agar skrip tidak bergantung pada statsmodels.
def durbin_watson(res):
    return float(np.sum(np.diff(res) ** 2) / np.sum(res ** 2))


def ljung_box(res, h=3):
    n = len(res); e = res - res.mean(); c0 = np.sum(e ** 2)
    rho = np.array([np.sum(e[k:] * e[:-k]) / c0 for k in range(1, h + 1)])
    Q = n * (n + 2) * np.sum(rho ** 2 / (n - np.arange(1, h + 1)))
    return float(Q), float(stats.chi2.sf(Q, h))


def breusch_pagan(res, yhat):
    # LM = n * R^2 dari regresi residual kuadrat terhadap nilai prediksi (versi Koenker)
    u = res ** 2; X = np.column_stack([np.ones_like(yhat), yhat])
    b, *_ = np.linalg.lstsq(X, u, rcond=None)
    r2 = 1 - np.sum((u - X @ b) ** 2) / np.sum((u - u.mean()) ** 2)
    LM = len(u) * r2
    return float(LM), float(stats.chi2.sf(LM, 1))

df = pd.read_csv(DATA_CSV); OUT = ANALISIS
yy, mm = df.periode.str.split("-", expand=True).astype(int).T.values
df["t"] = (yy - 2020) * 12 + (mm - 1)

lg = lambda t, K, r, tm: K / (1 + np.exp(-r * (t - tm)))
gp = lambda t, K, r, tm: K * np.exp(-np.exp(-r * (t - tm)))
bl = lambda t, K1, r1, tm1, K2, r2, tm2: lg(t, K1, r1, tm1) + lg(t, K2, r2, tm2)


def runs_test(res):
    s = np.sign(res); s = s[s != 0]
    n1, n2 = int((s > 0).sum()), int((s < 0).sum())
    runs = 1 + int((s[1:] != s[:-1]).sum())
    mu = 2 * n1 * n2 / (n1 + n2) + 1
    var = 2 * n1 * n2 * (2 * n1 * n2 - n1 - n2) / ((n1 + n2) ** 2 * (n1 + n2 - 1))
    z = (runs - mu) / np.sqrt(var)
    return {"runs": runs, "harapan": float(mu), "z": float(z), "p": float(2 * stats.norm.sf(abs(z)))}


def diagnosa(y, yhat):
    res = y - yhat
    sw = stats.shapiro(res)
    lbQ, lbp = ljung_box(res, 3)
    bpLM, bpp = breusch_pagan(res, yhat)
    return {
        "n": int(len(y)), "SSE": float(np.sum(res ** 2)),
        "Shapiro_W": float(sw.statistic), "Shapiro_p": float(sw.pvalue),
        "Durbin_Watson": float(durbin_watson(res)),
        "LjungBox_Q3": lbQ, "LjungBox_p": lbp,
        "BreuschPagan_LM": bpLM, "BreuschPagan_p": bpp,
        "Runs": runs_test(res),
    }


out = {}
for col, nama in [("pengguna_juta", "Pengguna"), ("merchant_juta", "Merchant")]:
    d = df.dropna(subset=[col]); t = d.t.values.astype(float); y = d[col].values
    r = {}
    p, _ = curve_fit(lg, t, y, p0=[y.max() * 1.3, .06, 40], maxfev=100000)
    r["Logistik"] = diagnosa(y, lg(t, *p))
    p, _ = curve_fit(gp, t, y, p0=[y.max() * 1.3, .04, 30], maxfev=100000)
    r["Gompertz"] = diagnosa(y, gp(t, *p))
    if nama == "Pengguna":
        p, _ = curve_fit(bl, t, y, p0=[59, .09, 36, 8, .3, 74],
                         bounds=([40, .01, 10, 2, .05, 70], [80, 1, 60, 60, 1.5, 90]), maxfev=200000)
        r["Bi-logistik"] = diagnosa(y, bl(t, *p))
        # uji F model bersarang: logistik (3 parameter) vs bi-logistik (6 parameter)
        s1, s2, n = r["Logistik"]["SSE"], r["Bi-logistik"]["SSE"], len(y)
        df1, df2 = 3, n - 6
        F = ((s1 - s2) / df1) / (s2 / df2)
        r["Uji_F_logistik_vs_bilogistik"] = {"F": float(F), "df1": df1, "df2": df2,
                                             "p": float(stats.f.sf(F, df1, df2)),
                                             "R2_parsial": float((s1 - s2) / s1)}
    out[nama] = r

json.dump(out, open(f"{OUT}/hasil_diagnostik_residual.json", "w"), indent=2)
for nama, r in out.items():
    print("\n==", nama)
    for m, v in r.items():
        if "Shapiro_W" in v:
            print(f"{m:12s} n={v['n']} W={v['Shapiro_W']:.3f} p={v['Shapiro_p']:.3f} | DW={v['Durbin_Watson']:.2f} "
                  f"| LB Q(3)={v['LjungBox_Q3']:.2f} p={v['LjungBox_p']:.3f} | BP={v['BreuschPagan_LM']:.2f} p={v['BreuschPagan_p']:.3f} "
                  f"| runs={v['Runs']['runs']} (E={v['Runs']['harapan']:.1f}) z={v['Runs']['z']:.2f} p={v['Runs']['p']:.3f}")
        else:
            print(m, v)

## 7. *Moving block bootstrap* (remedi autokorelasi)
Panjang blok 3, 1.000 ulangan. Langkah ini memerlukan beberapa menit.

In [ ]:
# ---- Remedi autokorelasi: moving block bootstrap residual (panjang blok 3) ----
rng = np.random.default_rng(2026)


def block_boot(f, p, t, y, bounds=None, B=1000, L=3, idx_K=(0,)):
    yhat = f(t, *p); res = y - yhat; n = len(y); hasil = []
    starts = np.arange(n - L + 1)
    for _ in range(B):
        e = np.concatenate([res[s:s + L] for s in rng.choice(starts, size=int(np.ceil(n / L)))])[:n]
        try:
            kw = {"bounds": bounds} if bounds else {}
            pb, _ = curve_fit(f, t, yhat + e, p0=p, maxfev=200000, **kw)
            hasil.append(sum(pb[i] for i in idx_K))
        except Exception:
            pass
    return [float(np.percentile(hasil, 2.5)), float(np.percentile(hasil, 97.5))], len(hasil)


blok = {}
for col, nama in [("pengguna_juta", "Pengguna"), ("merchant_juta", "Merchant")]:
    d = df.dropna(subset=[col]); t = d.t.values.astype(float); y = d[col].values
    if nama == "Pengguna":
        bnd = ([40, .01, 10, 2, .05, 70], [80, 1, 60, 60, 1.5, 90])
        p, _ = curve_fit(bl, t, y, p0=[59, .09, 36, 8, .3, 74], bounds=bnd, maxfev=200000)
        ci, nb = block_boot(bl, p, t, y, bounds=bnd, idx_K=(0, 3))
        ci1, _ = block_boot(bl, p, t, y, bounds=bnd, idx_K=(0,))
        blok[nama] = {"Bi-logistik_K_total": float(p[0] + p[3]), "CI95_blok": ci,
                      "Bi-logistik_K1": float(p[0]), "K1_CI95_blok": ci1, "replikasi_berhasil": nb}
    else:
        for m, f, p0 in [("Logistik", lg, [y.max() * 1.3, .06, 40]), ("Gompertz", gp, [y.max() * 1.3, .04, 30])]:
            p, _ = curve_fit(f, t, y, p0=p0, maxfev=100000)
            ci, nb = block_boot(f, p, t, y)
            blok.setdefault(nama, {})[m] = {"K": float(p[0]), "CI95_blok": ci, "replikasi_berhasil": nb}
out["Block_bootstrap_L3"] = blok
json.dump(out, open(f"{OUT}/hasil_diagnostik_residual.json", "w"), indent=2)
print("\n== Moving block bootstrap (L=3, B=1000)")
print(json.dumps(blok, indent=1))

## 8. Gambar (warna dan hitam-putih)

In [ ]:
"""Gambar 1 (pengguna) dan Gambar 2 (merchant) 300 dpi."""
import json, sys
import matplotlib
import matplotlib.pyplot as plt

D = ANALISIS  # folder analisis
import numpy as np, pandas as pd
DATA = DATA_CSV
lg = lambda t, K, r, tm: K / (1 + np.exp(-r * (t - tm)))
gp = lambda t, K, r, tm: K * np.exp(-np.exp(-r * (t - tm)))
df = pd.read_csv(DATA); yy, mm = df.periode.str.split("-", expand=True).astype(int).T.values
df["t"] = (yy - 2020) * 12 + (mm - 1)
hm = json.load(open(f"{D}/hasil_model.json")); hb = json.load(open(f"{D}/hasil_bilogistik.json"))
pl = hb["logistik_tunggal"]["param_K_r_tm"]; pb = hb["bi_logistik"]["param_K1_r1_tm1_K2_r2_tm2"]
ml = hm["Merchant"]["model"]["Logistik"]["param"]; mg = hm["Merchant"]["model"]["Gompertz"]["param"]
def build(col, t0, curves):
    rows = [{"t": t, "s": n, "v": float(f(t))} for t in range(t0, 132, 3) for n, f in curves]
    rows += [{"t": int(r.t), "s": "Data BI", "v": float(getattr(r, col))} for r in df.dropna(subset=[col]).itertuples()]
    return rows
DATASETS = {
    "pengguna": build("pengguna_juta", 23, [("Logistik satu gelombang", lambda t: lg(t, *pl)), ("Bi-logistik", lambda t: lg(t, *pb[:3]) + lg(t, *pb[3:]))]),
    "merchant": build("merchant_juta", 32, [("Logistik", lambda t: lg(t, *ml)), ("Gompertz", lambda t: gp(t, *mg))]),
}
BLUE, GREY, REF = "#2a6fdb", "#9aa0a6", "#6b7280"

def plot(fn_json, series_main, series_alt, ref_val, ref_label, target, out, title, tmin):
    rows = DATASETS[fn_json]
    yr = lambda t: 2020 + t / 12
    obs = [r for r in rows if r["s"] == "Data BI"]
    main = [r for r in rows if r["s"] == series_main]
    alt = [r for r in rows if r["s"] == series_alt]
    fig, ax = plt.subplots(figsize=(8, 4.6), dpi=300)
    ax.plot([yr(r["t"]) for r in alt], [r["v"] for r in alt], "--", color=GREY, lw=1.8, label=series_alt)
    ax.plot([yr(r["t"]) for r in main], [r["v"] for r in main], "-", color=BLUE, lw=2.2, label=series_main)
    ax.scatter([yr(r["t"]) for r in obs], [r["v"] for r in obs], color=BLUE, s=16, zorder=3, label="Data BI")
    ax.axhline(ref_val, color=REF, ls=(0, (5, 4)), lw=1)
    ax.text(yr(tmin) + 0.1, ref_val + 1.2, ref_label, color=REF, fontsize=8)
    ax.axvline(yr(obs[-1]["t"]), color=REF, ls=":", lw=1)
    ax.text(yr(obs[-1]["t"]) + 0.08, 2, "proyeksi", color=REF, fontsize=8)
    ax.scatter([2026.92], [target[0]], facecolors="none", edgecolors="black", s=40, zorder=4)
    ax.annotate(target[1], (2026.92, target[0]), xytext=(10, -14), textcoords="offset points", fontsize=8)
    ax.set_ylim(0, 80); ax.set_xlim(yr(tmin) - 0.2, 2031)
    ax.set_ylabel("juta"); ax.set_title(title, fontsize=10, loc="left")
    ax.spines[["top", "right"]].set_visible(False); ax.grid(axis="y", color="#e5e7eb")
    ax.legend(frameon=False, fontsize=8, loc="lower right")
    fig.tight_layout(); fig.savefig(out); plt.close(fig)

plot("pengguna", "Bi-logistik", "Logistik satu gelombang", 60.72,
     "Batas jenuh gelombang pertama: 60,7 juta", (70, "Target BI 2026: 70 juta"),
     f"{D}/Gambar1_pengguna_dua_gelombang.png",
     "Gambar 1. Pengguna QRIS: model satu gelombang meleset 6,1 juta pada Agustus 2026", 23)
plot("merchant", "Logistik", "Gompertz", 65,
     "Populasi UMKM: ±65 juta unit", (47, "Target BI 2026: 47 juta"),
     f"{D}/Gambar2_merchant_batas_jenuh.png",
     "Gambar 2. Merchant QRIS: 58–72% dari batas jenuh (logistik 65,8 juta, setara jumlah UMKM)", 32)
# Versi hitam-putih tanpa judul di dalam gambar (judul ditulis sebagai keterangan di bawah gambar)
BLUE, GREY, REF = "black", "#666666", "#444444"
plot("pengguna", "Bi-logistik", "Logistik satu gelombang", 60.72,
     "Batas jenuh gelombang pertama: 60,7 juta", (70, "Target BI 2026: 70 juta"),
     f"{D}/Gambar1_pengguna_dua_gelombang_bw.png", "", 23)
plot("merchant", "Logistik", "Gompertz", 65,
     "Populasi UMKM: ±65 juta unit", (47, "Target BI 2026: 47 juta"),
     f"{D}/Gambar2_merchant_batas_jenuh_bw.png", "", 32)
print("ok")

In [ ]:
from IPython.display import Image, display
for g in ["Gambar1_pengguna_dua_gelombang_bw.png", "Gambar2_merchant_batas_jenuh_bw.png"]:
    display(Image(filename=str(BASE / "analisis" / g), width=800))

## 9. Pencocokan dengan angka yang dilaporkan

Kolom **Cocok** bernilai `True` apabila hasil sama dengan angka yang dilaporkan dalam batas toleransi (estimasi titik ±0,06; selang kepercayaan bootstrap ±0,6 karena dapat sedikit berbeda antarversi NumPy/SciPy). Apabila ada `False`, angka yang dilaporkan harus mengikuti hasil terbaru.

In [ ]:
A = BASE / "analisis"
m = json.load(open(A / "hasil_model.json"))
b = json.load(open(A / "hasil_bilogistik.json"))
u = json.load(open(A / "hasil_uji_ketahanan_resmi.json"))
d = json.load(open(A / "hasil_diagnostik_residual.json"))
blok = d["Block_bootstrap_L3"]
P, M = m["Pengguna"]["model"], m["Merchant"]["model"]
pm = pd.read_csv(DATA_CSV)
akhir = pm.dropna(subset=["pengguna_juta"]).iloc[-1]

# (yang diperiksa, hasil Ibu, angka yang dilaporkan, toleransi)
cek = [
    ("K logistik pengguna", P["Logistik"]["K"], 65.2, .06),
    ("SK K logistik pengguna (bawah)", P["Logistik"]["K_CI95"][0], 63.5, .6),
    ("SK K logistik pengguna (atas)", P["Logistik"]["K_CI95"][1], 67.5, .6),
    ("K Gompertz pengguna", P["Gompertz"]["K"], 69.1, .06),
    ("K Bass pengguna", P["Bass"]["K"], 66.0, .06),
    ("MAPE holdout logistik pengguna (%)", P["Logistik"]["holdout_MAPE_%"], 4.62, .01),
    ("MAPE holdout Gompertz pengguna (%)", P["Gompertz"]["holdout_MAPE_%"], 3.69, .01),
    ("AICc logistik tunggal", b["logistik_tunggal"]["AICc"], 35.9, .06),
    ("AICc bi-logistik", b["bi_logistik"]["AICc"], -33.1, .06),
    ("K1 gelombang pertama", b["bi_logistik"]["K1_gelombang1"], 60.72, .01),
    ("K total bi-logistik", b["bi_logistik"]["K_total"], 75.40, .01),
    ("Proyeksi pengguna Des 2026", b["bi_logistik"]["proyeksi_Des2026"], 73.6, .06),
    ("K logistik merchant", M["Logistik"]["K"], 65.8, .06),
    ("K Gompertz merchant", M["Gompertz"]["K"], 82.7, .06),
    ("Uji resmi: K1 bi-logistik", u["Pengguna"]["Bi-logistik"]["K1"], 59.8, .06),
    ("Uji resmi: K total bi-logistik", u["Pengguna"]["Bi-logistik"]["K_total"], 70.4, .06),
    ("Uji resmi: AICc bi-logistik", u["Pengguna"]["Bi-logistik"]["AICc"], -18.1, .06),
    ("Uji resmi: K logistik merchant", u["Merchant"]["Logistik"]["K"], 81.6, .06),
    ("Uji F logistik vs bi-logistik", d["Pengguna"]["Uji_F_logistik_vs_bilogistik"]["F"], 98.85, .01),
    ("Shapiro-Wilk W bi-logistik", d["Pengguna"]["Bi-logistik"]["Shapiro_W"], 0.973, .001),
    ("Durbin-Watson logistik pengguna", d["Pengguna"]["Logistik"]["Durbin_Watson"], 0.45, .006),
    ("Durbin-Watson bi-logistik", d["Pengguna"]["Bi-logistik"]["Durbin_Watson"], 0.94, .006),
    ("Blok: SK K1 (bawah)", blok["Pengguna"]["K1_CI95_blok"][0], 59.5, .6),
    ("Blok: SK K1 (atas)", blok["Pengguna"]["K1_CI95_blok"][1], 61.7, .6),
    ("Blok: SK K logistik merchant (bawah)", blok["Merchant"]["Logistik"]["CI95_blok"][0], 58.3, .6),
    ("Blok: SK K logistik merchant (atas)", blok["Merchant"]["Logistik"]["CI95_blok"][1], 79.5, .6),
    ("Pengguna Agustus 2026", akhir.pengguna_juta, 69.32, .001),
    ("Merchant Agustus 2026", akhir.merchant_juta, 47.11, .001),
]
tabel = pd.DataFrame(cek, columns=["Yang diperiksa", "Hasil", "Dilaporkan", "Toleransi"])
tabel["Hasil"] = tabel["Hasil"].astype(float).round(3)
tabel["Cocok"] = (tabel["Hasil"] - tabel["Dilaporkan"]).abs() <= tabel["Toleransi"]
tabel.to_csv(BASE / "hasil_verifikasi_ulang.csv", index=False)
print(f"{tabel.Cocok.sum()} dari {len(tabel)} angka cocok (tersimpan di hasil_verifikasi_ulang.csv)")
tabel

## 10. Pemeriksaan tambahan: apakah data empiris mendukung pola kurva S?

Bagian ini **menambahkan pemeriksaan tanpa mengganti model utama**. Karena observasi QRIS tidak selalu berjarak sama, laju pertumbuhan dihitung sebagai perubahan jumlah per bulan di antara dua observasi aktual. Turunan kedua dihitung secara numerik dari laju tersebut dan bersifat eksploratif; hasilnya tidak boleh diperlakukan sebagai uji formal tunggal atas bentuk S.

In [ ]:
# Pemeriksaan empiris bentuk S: pertumbuhan antarobservasi dan perubahan laju
import numpy as np
import pandas as pd
from pathlib import Path

df_s = pd.read_csv(DATA_CSV)
yy, mm = df_s["periode"].str.split("-", expand=True).astype(int).T.values
df_s["t"] = (yy - 2020) * 12 + (mm - 1)

def empirical_growth_check(data, col, label):
    d = data[["periode", "t", col]].dropna().sort_values("t").copy()
    d["delta_t_bulan"] = d["t"].diff()
    d["delta_N_juta"] = d[col].diff()
    d["laju_juta_per_bulan"] = d["delta_N_juta"] / d["delta_t_bulan"]
    # Laju ditempatkan pada titik tengah dua observasi; turunan kedua pendekatan.
    d["t_tengah"] = (d["t"] + d["t"].shift(1)) / 2
    d["perubahan_laju_per_bulan2"] = d["laju_juta_per_bulan"].diff() / d["t_tengah"].diff()
    d["fase_laju"] = np.select(
        [d["perubahan_laju_per_bulan2"] > 0, d["perubahan_laju_per_bulan2"] < 0],
        ["laju sedang meningkat", "laju sedang menurun"],
        default="belum dapat diklasifikasikan"
    )
    out = BASE / "analisis" / f"diagnostik_empiris_kurva_S_{label.lower()}.csv"
    d.to_csv(out, index=False)
    print(f"\\n{label}: observasi={len(d)}; periode={d.periode.iloc[0]} s.d. {d.periode.iloc[-1]}")
    print("Laju pertumbuhan antarobservasi (juta/bulan), lima baris terakhir:")
    display(d[["periode", col, "delta_t_bulan", "laju_juta_per_bulan",
               "perubahan_laju_per_bulan2", "fase_laju"]].tail())
    print("Disimpan:", out)
    return d

growth_users = empirical_growth_check(df_s, "pengguna_juta", "Pengguna")
growth_merchants = empirical_growth_check(df_s, "merchant_juta", "Merchant")

## 11. Sensitivitas estimasi K terhadap cut-off data

Model diestimasi ulang dengan tiga titik akhir data yang tersedia (jika jumlah observasi mencukupi). Tujuannya mengukur apakah K berubah ketika observasi terbaru dimasukkan. Nilai K yang berubah tajam berarti estimasi batas jenuh belum stabil; nilai K tidak otomatis berarti target populasi yang pasti.

In [ ]:
# Sensitivitas K untuk model utama; model asli tidak diubah
import json
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit

df_k = pd.read_csv(DATA_CSV)
yy, mm = df_k["periode"].str.split("-", expand=True).astype(int).T.values
df_k["t"] = (yy - 2020) * 12 + (mm - 1)

def f_logistic(t, K, r, tm):
    return K / (1 + np.exp(-r * (t - tm)))
def f_gompertz(t, K, r, tm):
    return K * np.exp(-np.exp(-r * (t - tm)))
def f_bass(t, m, p, q):
    e = np.exp(-(p + q) * t)
    return m * (1 - e) / (1 + (q / p) * e)
def f_bilogistic(t, K1, r1, tm1, K2, r2, tm2):
    return f_logistic(t, K1, r1, tm1) + f_logistic(t, K2, r2, tm2)

cutoffs = sorted(set([x for x in ["2025-12", "2026-06", df_k["periode"].max()] if x in set(df_k["periode"])]))
# Jika label cut-off tidak persis ada di dataset, gunakan observasi terakhir sebelum label tersebut.
if len(cutoffs) < 3:
    for cutoff in ["2025-12", "2026-06", df_k["periode"].max()]:
        valid = df_k.loc[df_k["periode"] <= cutoff, "periode"]
        if len(valid):
            actual = valid.max()
            if actual not in cutoffs:
                cutoffs.append(actual)
cutoffs = sorted(set(cutoffs))

sensitivity_rows = []
for col, label in [("pengguna_juta", "Pengguna"), ("merchant_juta", "Merchant")]:
    for cutoff in cutoffs:
        d = df_k.loc[(df_k["periode"] <= cutoff) & df_k[col].notna()].copy()
        if len(d) < 8:
            continue
        t, y = d["t"].to_numpy(float), d[col].to_numpy(float)
        configs = [
            ("Logistik", f_logistic, [max(y.max()*1.15, 1), .06, 40],
             ([1, 1e-4, -60], [1000, 1, 200]), (0,)),
            ("Gompertz", f_gompertz, [max(y.max()*1.15, 1), .04, 30],
             ([1, 1e-4, -60], [1000, 1, 200]), (0,)),
            ("Bass", f_bass, [max(y.max()*1.15, 1), .005, .05],
             ([1, 1e-6, 1e-4], [1000, .5, 1]), (0,))
        ]
        for name, func, p0, bounds, kidx in configs:
            try:
                p, _ = curve_fit(func, t, y, p0=p0, bounds=bounds, maxfev=200000)
                pred = func(t, *p)
                sse = float(np.sum((y-pred)**2))
                sensitivity_rows.append({"Variabel": label, "Cutoff aktual": d["periode"].iloc[-1],
                    "Model": name, "n": len(y), "K": float(p[0]), "RMSE_in_sample": float(np.sqrt(sse/len(y))),
                    "Observasi terakhir": float(y[-1]), "Observasi/K": float(y[-1]/p[0])})
            except Exception as e:
                sensitivity_rows.append({"Variabel": label, "Cutoff aktual": d["periode"].iloc[-1],
                    "Model": name, "n": len(y), "K": np.nan, "RMSE_in_sample": np.nan,
                    "Observasi terakhir": float(y[-1]), "Observasi/K": np.nan, "Catatan": str(e)[:160]})
        if label == "Pengguna" and len(y) >= 12:
            try:
                bounds = ([40, .01, 10, 2, .05, 70], [80, 1, 60, 60, 1.5, 90])
                p, _ = curve_fit(f_bilogistic, t, y, p0=[59, .09, 36, 8, .3, 74],
                                 bounds=bounds, maxfev=200000)
                pred = f_bilogistic(t, *p)
                sensitivity_rows.append({"Variabel": label, "Cutoff aktual": d["periode"].iloc[-1],
                    "Model": "Bi-logistik", "n": len(y), "K": float(p[0]+p[3]),
                    "RMSE_in_sample": float(np.sqrt(np.mean((y-pred)**2))),
                    "Observasi terakhir": float(y[-1]), "Observasi/K": float(y[-1]/(p[0]+p[3])),
                    "K1": float(p[0]), "K2": float(p[3]),
                    "Infleksi 1 t": float(p[2]), "Infleksi 2 t": float(p[5])})
            except Exception as e:
                print("Bi-logistik gagal untuk cut-off", d["periode"].iloc[-1], ":", e)

df_sensitivity_K = pd.DataFrame(sensitivity_rows)
display(df_sensitivity_K.sort_values(["Variabel", "Model", "Cutoff aktual"]))
out_k = BASE / "analisis" / "sensitivitas_K_cutoff.csv"
df_sensitivity_K.to_csv(out_k, index=False)
print("Disimpan:", out_k)
print("Catatan: cut-off berbeda berarti ukuran sampel berbeda; bandingkan arah dan besar perubahan K, bukan hanya RMSE.")

## 12. Validasi expanding-window (uji prediksi lintas waktu)

Validasi ini melatih model hanya pada data sebelum suatu cut-off lalu memprediksi observasi setelahnya. Tiga cut-off digunakan bila tersedia. MAPE/RMSE dihitung hanya pada data uji. Untuk data yang jarang/tidak berjarak bulanan, pembagian tetap mengikuti tanggal observasi aktual.

In [ ]:
# Validasi expanding-window: prediksi masa depan tanpa memasukkan data uji saat fitting
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit

def run_expanding_validation(data, col, label):
    d = data[["periode", "t", col]].dropna().sort_values("t").copy()
    # Cut-off eksplisit, dipetakan ke observasi terakhir yang tidak melewatinya.
    requested = ["2023-12", "2024-12", "2025-06"]
    rows = []
    for req in requested:
        train = d[d["periode"] <= req]
        test = d[d["periode"] > req]
        if len(train) < 8 or len(test) == 0:
            continue
        ttr, ytr = train["t"].to_numpy(float), train[col].to_numpy(float)
        tte, yte = test["t"].to_numpy(float), test[col].to_numpy(float)
        for name, func, p0, bounds in [
            ("Logistik", f_logistic, [max(ytr.max()*1.3, 1), .06, 40],
             ([1, 1e-4, -60], [1000, 1, 200])),
            ("Gompertz", f_gompertz, [max(ytr.max()*1.3, 1), .04, 30],
             ([1, 1e-4, -60], [1000, 1, 200])),
            ("Bass", f_bass, [max(ytr.max()*1.3, 1), .005, .05],
             ([1, 1e-6, 1e-4], [1000, .5, 1]))
        ]:
            try:
                p, _ = curve_fit(func, ttr, ytr, p0=p0, bounds=bounds, maxfev=200000)
                yh = func(tte, *p)
                err = yte-yh
                rows.append({"Variabel": label, "Cutoff diminta": req,
                    "Cutoff aktual train": train["periode"].iloc[-1],
                    "Periode uji": f'{test["periode"].iloc[0]} s.d. {test["periode"].iloc[-1]}',
                    "Model": name, "n_train": len(train), "n_test": len(test),
                    "RMSE_test": float(np.sqrt(np.mean(err**2))),
                    "MAPE_test_%": float(np.mean(np.abs(err/yte))*100),
                    "Bias_test_juta": float(np.mean(yh-yte))})
            except Exception as e:
                rows.append({"Variabel": label, "Cutoff diminta": req,
                    "Cutoff aktual train": train["periode"].iloc[-1],
                    "Periode uji": f'{test["periode"].iloc[0]} s.d. {test["periode"].iloc[-1]}',
                    "Model": name, "n_train": len(train), "n_test": len(test),
                    "RMSE_test": np.nan, "MAPE_test_%": np.nan,
                    "Bias_test_juta": np.nan, "Catatan": str(e)[:160]})
    return rows

rolling_rows = []
for col, label in [("pengguna_juta", "Pengguna"), ("merchant_juta", "Merchant")]:
    rolling_rows.extend(run_expanding_validation(df_k.assign(t=(df_k["periode"].str[:4].astype(int)-2020)*12+
        (df_k["periode"].str[5:7].astype(int)-1)), col, label))
df_rolling = pd.DataFrame(rolling_rows)
display(df_rolling)
out_roll = BASE / "analisis" / "validasi_expanding_window.csv"
df_rolling.to_csv(out_roll, index=False)
print("Disimpan:", out_roll)
print("Interpretasi: MAPE yang rendah dan relatif stabil lintas cut-off lebih meyakinkan daripada satu holdout saja; n_test kecil perlu disebutkan.")

## 13. Plot diagnostik: data aktual, kurva fitted, dan titik infleksi

Grafik diagnostik berikut dibuat terpisah dari grafik utama agar hasil tambahan tidak menimpa gambar yang sudah ada. Garis infleksi adalah infleksi teoretis model terpilih, bukan selalu titik infleksi yang dapat dibuktikan langsung dari observasi.

In [ ]:
# Grafik diagnostik tambahan, disimpan terpisah dari Gambar 1 dan 2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import json

df_plot = pd.read_csv(DATA_CSV)
yy, mm = df_plot["periode"].str.split("-", expand=True).astype(int).T.values
df_plot["t"] = (yy-2020)*12 + (mm-1)
res_model = json.load(open(BASE / "analisis" / "hasil_model.json", encoding="utf-8"))
res_bi = json.load(open(BASE / "analisis" / "hasil_bilogistik.json", encoding="utf-8"))

def diagnostic_plot(col, label):
    d = df_plot[["periode", "t", col]].dropna().sort_values("t")
    t, y = d["t"].to_numpy(float), d[col].to_numpy(float)
    grid = np.linspace(t.min(), max(t.max()+24, t.max()), 300)
    plt.figure(figsize=(9, 5))
    plt.scatter(2020+t/12, y, label="Data aktual", zorder=3)
    curves = []
    for name in ["Logistik", "Gompertz", "Bass"]:
        p = res_model[label]["model"][name]["param"]
        func = {"Logistik": f_logistic, "Gompertz": f_gompertz, "Bass": f_bass}[name]
        plt.plot(2020+grid/12, func(grid, *p), label=name)
        if name == "Logistik":
            inflection = p[2]
            plt.axvline(2020+inflection/12, linestyle=":", linewidth=1,
                        label="Infleksi logistik")
    if label == "Pengguna":
        p = res_bi["bi_logistik"]["param_K1_r1_tm1_K2_r2_tm2"]
        plt.plot(2020+grid/12, f_bilogistic(grid, *p), linestyle="--", linewidth=2,
                 label="Bi-logistik")
        plt.axvline(2020+p[2]/12, linestyle=":", linewidth=1, label="Infleksi 1 bi-logistik")
        plt.axvline(2020+p[5]/12, linestyle=":", linewidth=1, label="Infleksi 2 bi-logistik")
    plt.axvline(2020+t.max()/12, linestyle="-.", linewidth=1, label="Observasi terakhir")
    plt.xlabel("Tahun")
    plt.ylabel("Jumlah (juta)")
    plt.title(f"{label}: data aktual dan model kurva S")
    plt.legend(fontsize=8, ncol=2)
    plt.tight_layout()
    out = BASE / "analisis" / f"diagnostik_kurva_S_{label.lower()}.png"
    plt.savefig(out, dpi=300, bbox_inches="tight")
    plt.show()
    print("Disimpan:", out)

diagnostic_plot("pengguna_juta", "Pengguna")
diagnostic_plot("merchant_juta", "Merchant")

## 14. Cara membaca hasil tambahan dan batas interpretasi

1. **Laju empiris:** laju pertumbuhan antarobservasi dapat naik lalu turun, tetapi karena data tidak selalu bulanan dan memiliki noise, ini adalah indikasi deskriptif, bukan bukti tunggal kurva S.
2. **Sensitivitas K:** bila K berubah besar ketika cut-off bergeser, tuliskan bahwa saturation belum teridentifikasi stabil.
3. **Expanding-window:** bandingkan RMSE, MAPE, dan bias di setiap periode uji; jumlah titik uji yang kecil harus dinyatakan.
4. **Titik infleksi:** titik infleksi fitted merupakan sifat matematis kurva model. Jangan menyebutnya sebagai tanggal perubahan perilaku yang pasti tanpa dukungan data tambahan.
5. **K bukan target pasti:** estimasi asimtotik adalah hasil model dan sangat dipengaruhi rentang data, spesifikasi model, dan asumsi.
6. Semua keluaran tambahan disimpan sebagai berkas baru di folder `analisis`; kode, model, dan berkas gambar utama sebelumnya tidak ditimpa.